In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
import time
import os

In [6]:
def getDateofNewsfromCSV(save_path, symbol):
    if os.path.exists(save_path):
        df = pd.read_csv(f"{save_path}")
        print(f"{symbol}news.csv file exist!")
        if not df.empty:
            df["Date"] = pd.to_datetime(df["Date"])
            save_date_ts = df["Date"].iloc[0]
            save_date = save_date_ts.date()
            return save_date, df
            
        else:
            return None, df
    else:
        print(f"{symbol}news.csv file not exist!")
        return None, None

def fetchFullContent(link, date, headline, data, symbol, i):
    print(f"{(i+1)} ✓ Fetching {symbol}...", end=" ")
    try:
        response = requests.get(link, timeout=10)
        response.raise_for_status()
        article_soup = BeautifulSoup(response.content, "html.parser")
        content_div = article_soup.find("div", id="newsdetail-content")
        content = content_div.get_text(separator="\n", strip=True) if content_div else ""
    except Exception as e:
        print(f"Failed to fetch {link} for {headline}: {e}")
        content = ""

    data["Date"].append(date)
    data["Headline"].append(headline)
    data["Link"].append(link)
    data["Full Content"].append(content)
    print(f"✓ Data append...", end="   ")
    return 

query = "NEPSECompanyExtractor"
companyDetails = pd.read_csv(f"../DATA-HTML-STOCK/NEPSECompany/{query}.csv")
symbols = companyDetails['Symbol']



for index, symbol in enumerate(symbols[3:]):
    
    save_date = None
    data = {'Date': [], 'Headline': [], 'Link': [], 'Full Content': []}
    
    html_path = f"../DATA-HTML-STOCK/webScrapped-htmlfiles/news-WEB-SCRAP-htmlfile/{symbol}news.html"
    save_path = f"../DATA-HTML-STOCK/NEPSENEWS/{symbol}news.csv"
    
    save_date, df = getDateofNewsfromCSV(save_path, symbol)
    print(f"✓ Parsing HTML {symbol}...\n")
    if os.path.exists(html_path):
        with open(html_path, encoding="utf-8") as f:
                soup = BeautifulSoup(f.read(), "html.parser")
    else:
        print(f"{symbol}news.html file not exist")
        continue


    soupbdy = soup.find("tbody")
    if soupbdy is None:
        print(f"No news table found for {symbol}")
        continue

    rows = soupbdy.find_all("tr")

    for i, row in enumerate(rows):
        
        cols = row.find_all("td")
        date = pd.to_datetime(cols[0].get_text(strip=True)).date()
        headline = cols[1].get_text(strip=True)
        link = cols[1].find("a")["href"].strip()

        
        if save_date is not None:
            if date<=save_date:
                break

        fetchFullContent(link, date, headline, data, symbol, i)
        time.sleep(1)

    new_df = pd.DataFrame(data)
    if save_date is not None:
        if not new_df.empty:
            final_df = pd.concat([new_df, df], ignore_index=True)
            final_df.to_csv(save_path, index=False)
        else:
            print(f"No new news for {symbol}")
    else:       
        new_df.to_csv(save_path, index=False)  
    print(f"✓ Parsing HTML {symbol}... Done")
    print(f"✓ {(len(symbols)-index)} remain to parse...\n")
    
print("All Done!")

HATHPOnews.csv file not exist!
✓ Parsing HTML HATHPO...

HATHPOnews.html file not exist
AKPLnews.csv file not exist!
✓ Parsing HTML AKPL...

1 ✓ Fetching AKPL... ✓ Data append...   2 ✓ Fetching AKPL... ✓ Data append...   3 ✓ Fetching AKPL... ✓ Data append...   4 ✓ Fetching AKPL... ✓ Data append...   5 ✓ Fetching AKPL... ✓ Data append...   6 ✓ Fetching AKPL... ✓ Data append...   7 ✓ Fetching AKPL... ✓ Data append...   8 ✓ Fetching AKPL... ✓ Data append...   9 ✓ Fetching AKPL... ✓ Data append...   10 ✓ Fetching AKPL... ✓ Data append...   11 ✓ Fetching AKPL... ✓ Data append...   12 ✓ Fetching AKPL... ✓ Data append...   13 ✓ Fetching AKPL... ✓ Data append...   14 ✓ Fetching AKPL... ✓ Data append...   15 ✓ Fetching AKPL... ✓ Data append...   16 ✓ Fetching AKPL... ✓ Data append...   17 ✓ Fetching AKPL... ✓ Data append...   18 ✓ Fetching AKPL... ✓ Data append...   19 ✓ Fetching AKPL... ✓ Data append...   20 ✓ Fetching AKPL... ✓ Data append...   21 ✓ Fetching AKPL... ✓ Data append...   22 ✓ Fe

NotImplementedError: date not yet supported on Timestamps which are outside the range of Python's standard library. 

In [ ]:
def getDateofNewsfromCSV(save_path, symbol):
    if os.path.exists(save_path):
        df = pd.read_csv(save_path)
        print(f"{symbol}news.csv file exists!")

        if not df.empty:
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            df = df.dropna(subset=["Date"])

            if not df.empty:
                save_date = df["Date"].iloc[0]
                return save_date, df

        return None, df

    else:
        print(f"{symbol}news.csv file not exist!")
        return None, None


def fetchFullContent(link, date, headline, data, symbol, i):
    print(f"{i+1} ✓ Fetching {symbol}...", end=" ")

    try:
        response = requests.get(link, timeout=10)
        response.raise_for_status()
        article_soup = BeautifulSoup(response.content, "html.parser")

        content_div = article_soup.find("div", id="newsdetail-content")
        content = (
            content_div.get_text(separator="\n", strip=True)
            if content_div else ""
        )

    except Exception as e:
        print(f"Failed to fetch {link}: {e}")
        content = ""

    data["Date"].append(date)
    data["Headline"].append(headline)
    data["Link"].append(link)
    data["Full Content"].append(content)

    print("✓ Data appended")
    return



query = "NEPSECompanyExtractor"
companyDetails = pd.read_csv(f"../DATA-HTML-STOCK/NEPSECompany/{query}.csv")
symbols = companyDetails["Symbol"]


for index, symbol in enumerate(symbols[21:]):

    data = {"Date": [],"Headline": [],"Link": [],"Full Content": []}

    html_path = f"../DATA-HTML-STOCK/webScrapped-htmlfiles/news-WEB-SCRAP-htmlfile/{symbol}news.html" 
    save_path = f"../DATA-HTML-STOCK/NEPSENEWS/{symbol}news.csv"

    save_date, old_df = getDateofNewsfromCSV(save_path, symbol)

    print(f"\n✓ Parsing HTML {symbol}...")

    if not os.path.exists(html_path):
        print(f"{symbol}news.html file not exist")
        continue

    with open(html_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    soupbdy = soup.find("tbody")

    if soupbdy is None:
        print(f"No news table found for {symbol}")
        continue

    rows = soupbdy.find_all("tr")

    for i, row in enumerate(rows):

        cols = row.find_all("td")

        if len(cols) < 2:
            continue

        date_text = cols[0].get_text(strip=True)
        parsed_date = pd.to_datetime(date_text, errors="coerce")

        if pd.isna(parsed_date):
            print(f"⚠ Skipping invalid date: {date_text}")
            continue

        date = parsed_date

        headline = cols[1].get_text(strip=True)

        link_tag = cols[1].find("a")
        if not link_tag or not link_tag.get("href"):
            continue

        link = link_tag["href"].strip()

        if save_date is not None:
            if date <= save_date:
                break

        fetchFullContent(link, date, headline, data, symbol, i)
        time.sleep(1)

    new_df = pd.DataFrame(data)

    if save_date is not None:

        if not new_df.empty:
            final_df = pd.concat(
                [new_df, old_df],
                ignore_index=True
            )
            final_df = final_df.sort_values(
                by="Date", ascending=False
            )
            final_df.to_csv(save_path, index=False)
            print(f"✓ Updated {symbol}news.csv")

        else:
            print(f"No new news for {symbol}")

    else:
        if not new_df.empty:
            new_df = new_df.sort_values(
                by="Date", ascending=False
            )
            new_df.to_csv(save_path, index=False)
            print(f"✓ Created {symbol}news.csv")

    print(f"✓ {len(symbols) - index} remain to parse...\n")


print("All Done!")

CFCLnews.csv file exists!

✓ Parsing HTML CFCL...
No new news for CFCL
✓ 300 remain to parse...

CFCLPOnews.csv file not exist!

✓ Parsing HTML CFCLPO...
CFCLPOnews.html file not exist
CCBLnews.csv file not exist!

✓ Parsing HTML CCBL...
1 ✓ Fetching CCBL... ✓ Data appended
2 ✓ Fetching CCBL... ✓ Data appended
3 ✓ Fetching CCBL... ✓ Data appended
4 ✓ Fetching CCBL... ✓ Data appended
5 ✓ Fetching CCBL... ✓ Data appended
6 ✓ Fetching CCBL... ✓ Data appended
7 ✓ Fetching CCBL... ✓ Data appended
8 ✓ Fetching CCBL... ✓ Data appended
9 ✓ Fetching CCBL... ✓ Data appended
10 ✓ Fetching CCBL... ✓ Data appended
11 ✓ Fetching CCBL... ✓ Data appended
12 ✓ Fetching CCBL... ✓ Data appended
13 ✓ Fetching CCBL... ✓ Data appended
14 ✓ Fetching CCBL... ✓ Data appended
15 ✓ Fetching CCBL... ✓ Data appended
16 ✓ Fetching CCBL... ✓ Data appended
17 ✓ Fetching CCBL... ✓ Data appended
18 ✓ Fetching CCBL... ✓ Data appended
19 ✓ Fetching CCBL... ✓ Data appended
20 ✓ Fetching CCBL... ✓ Data appended
21 ✓ Fetchi